# 14c — Robustesse hors-modèle sur la barrière : là où la fragilité apparaît enfin

## L'idée, après 14 et 14b

Sur le call vanille, le deep hedger restait robuste hors modèle, même aveugle à la volatilité. On avait dégagé le principe : **le sur-ajustement au simulateur frappe les paramètres qui pilotent la politique optimale et que le réseau n'observe pas.** Pour le vanille, la quantité pilote (la proba de finir ITM) bouge à peine avec le régime : 62% → 55% → 67%. Donc rien à sur-ajuster.

La barrière est l'exact opposé. Sa quantité pilote, la **probabilité de knock-out**, explose avec le régime (chiffres sous la mesure de pricing risque-neutre) :

| Régime | proba de knock-out | prime de la barrière |
|---|---|---|
| P0 | ~10% | 5.48 |
| S2 vol 30% | **~30%** | **2.65** |

Toute la couverture consiste à décider *à quel point couper la position en approchant le seuil*, et ça dépend de la proba de knock, donc du niveau de vol. On teste donc deux réseaux : un qui **voit** la variance $v$, un **aveugle** à $v$. Prédiction : le réseau aveugle, calibré sur le régime de knock de P0, devrait se casser sur S2 (il ne peut pas savoir que le knock a triplé), et la randomisation devrait le réparer. Le même aveuglement n'avait quasiment rien fait au vanille (14b) : c'est la démonstration que **la robustesse est une propriété du produit**.

On couvre avec le **sous-jacent seul** pour isoler l'effet du régime (le moteur est la proba de knock, pas le choix d'instrument).


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm

torch.manual_seed(0)
S0, K, mu, r, T = 100., 100., 0.05, 0.02, 1.0
n, cost, alpha = 63, 0.01, 0.95; dt = T/n; Bar = 130.0

def bs_uo_call(S,X,H,Tt,r,sig):
    S=np.asarray(S,float); srt=sig*np.sqrt(Tt); m_=(r-0.5*sig**2)/sig**2
    x1=np.log(S/X)/srt+(1+m_)*srt; x2=np.log(S/H)/srt+(1+m_)*srt
    y1=np.log(H**2/(S*X))/srt+(1+m_)*srt; y2=np.log(H/S)/srt+(1+m_)*srt
    A=S*norm.cdf(x1)-X*np.exp(-r*Tt)*norm.cdf(x1-srt); B=S*norm.cdf(x2)-X*np.exp(-r*Tt)*norm.cdf(x2-srt)
    C=S*(H/S)**(2*(m_+1))*norm.cdf(-y1)-X*np.exp(-r*Tt)*(H/S)**(2*m_)*norm.cdf(-y1+srt)
    D=S*(H/S)**(2*(m_+1))*norm.cdf(-y2)-X*np.exp(-r*Tt)*(H/S)**(2*m_)*norm.cdf(-y2+srt)
    return np.where(S>=H,0.,A-B+C-D)
def bs_uo_delta(S,X,H,Tt,r,sig,h=0.5): return (bs_uo_call(S+h,X,H,Tt,r,sig)-bs_uo_call(S-h,X,H,Tt,r,sig))/(2*h)
def bs_price(S,K,tau,r,s): d1=(np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)); return S*norm.cdf(d1)-K*np.exp(-r*tau)*norm.cdf(d1-s*np.sqrt(tau))
from scipy.optimize import brentq
def cvar(p,a=0.95): l=-p; return l[l>=np.quantile(l,a)].mean()

def sim_np(par, m, seed, drift):
    v0,kappa,theta,xi,rho=par; rng=np.random.default_rng(seed)
    S=np.empty((m,n+1)); v=np.empty((m,n+1)); S[:,0]=S0; v[:,0]=v0
    for k in range(n):
        Z1=rng.standard_normal(m); Z2=rho*Z1+np.sqrt(1-rho**2)*rng.standard_normal(m)
        vk=np.maximum(v[:,k],0.)
        v[:,k+1]=np.maximum(v[:,k]+kappa*(theta-vk)*dt+xi*np.sqrt(vk*dt)*Z2,0.)
        S[:,k+1]=S[:,k]*np.exp((drift-0.5*vk)*dt+np.sqrt(vk*dt)*Z1)
    return S,v

P0=(0.04,2.0,0.04,0.3,-0.7)
scen={'P0':P0,'S1 xi=0.6':(0.04,2,0.04,0.6,-0.7),'S2 vol30%':(0.09,2,0.09,0.3,-0.7),
      'S3 rho=-0.3':(0.04,2,0.04,0.3,-0.3),'S4 crash':(0.04,2,0.04,0.5,-0.9)}
times=np.linspace(0,T,n+1)

# prime de barriere (MC risque-neutre) + reference delta BS-barriere, par scenario
ref={}
for name,par in scen.items():
    Sq,_=sim_np(par,120000,1,r); kn=(Sq.max(axis=1)>=Bar)
    prem=float(np.exp(-r*T)*np.where(~kn,np.maximum(Sq[:,-1]-K,0.),0.).mean())
    pv=bs_price  # placeholder
    # vol implicite ATM recalibree (prix vanille Heston approxime par MC risque-neutre)
    premvan=float(np.exp(-r*T)*np.maximum(Sq[:,-1]-K,0.).mean())
    sig=brentq(lambda s: bs_price(S0,K,T,r,s)-premvan,1e-3,2.0)
    S,v=sim_np(par,60000,7,mu); alive=np.cumprod((S<Bar).astype(float),axis=1)
    payoff=np.where(alive[:,-1]>0,np.maximum(S[:,-1]-K,0.),0.)
    cash=np.full(60000,prem); pos=np.zeros(60000)
    for k in range(n):
        tau=max(T-times[k],1e-3); tgt=bs_uo_delta(S[:,k],K,Bar,tau,r,sig)*alive[:,k]
        tr=tgt-pos; cash-=tr*S[:,k]+cost*np.abs(tr)*S[:,k]; pos=tgt; cash*=np.exp(r*dt)
    ref[name]=dict(prem=prem, knock=kn.mean(), delta=cvar(cash+pos*S[:,-1]-payoff))
    print(f"{name:12s} knock={kn.mean():5.1%}  prime={prem:5.2f}  classique(deltaBS)={ref[name]['delta']:6.2f}")
prem_P0=ref['P0']['prem']


## Le hedger de barrière (sous-jacent seul), version « voit $v$ » et version « aveugle »

Même recette que le notebook 14 : Monte Carlo frais à chaque époque, perte CVaR empirique directe. La seule différence entre les deux réseaux est la présence ou non de $v$ dans les features. Les deux voient la **distance à la barrière** et le **statut vivant**, indispensables pour un passif à barrière.


In [ ]:
def paths_t(par, m):
    v0,kappa,theta,xi,rho=par
    S=torch.full((m,),S0); v=torch.full((m,),float(v0)); Ss=[S]; vs=[v]
    for k in range(n):
        Z1=torch.randn(m); Z2=rho*Z1+np.sqrt(1-rho**2)*torch.randn(m)
        vk=torch.clamp(v,min=0.)
        v=torch.clamp(v+kappa*(theta-vk)*dt+xi*torch.sqrt(vk*dt)*Z2,min=0.)
        S=S*torch.exp((mu-0.5*vk)*dt+torch.sqrt(vk*dt)*Z1); Ss.append(S); vs.append(v)
    return torch.stack(Ss,1), torch.stack(vs,1)

def cvar_torch(L,a=0.95): var=torch.quantile(L,a); return L[L>=var].mean()

class Net(torch.nn.Module):
    def __init__(self, d, h=48):
        super().__init__()
        self.net=torch.nn.Sequential(torch.nn.Linear(d,h),torch.nn.ReLU(),
                                     torch.nn.Linear(h,h),torch.nn.ReLU(),torch.nn.Linear(h,1))
    def forward(self,x): return self.net(x).squeeze(-1)

def barrier_pnl(net, S, v, premium, use_v):
    m=S.shape[0]; cash=torch.full((m,),premium); pos=torch.zeros(m); alive=torch.ones(m)
    for k in range(n):
        alive=alive*(S[:,k]<Bar).float()               # 0 une fois la barriere franchie
        tau=float(T-k*dt); dist=(Bar-S[:,k])/S0
        if use_v:
            feat=torch.stack([torch.log(S[:,k]/K),torch.full((m,),tau),pos,v[:,k],dist,alive],1)
        else:
            feat=torch.stack([torch.log(S[:,k]/K),torch.full((m,),tau),pos,dist,alive],1)
        d=net(feat); tr=d-pos
        cash=cash-tr*S[:,k]-cost*torch.abs(tr)*S[:,k]; cash=cash*np.exp(r*dt); pos=d
    alive=alive*(S[:,-1]<Bar).float()
    payoff=torch.clamp(S[:,-1]-K,min=0.)*alive
    return cash+pos*S[:,-1]-payoff

def train(use_v, sampler, epochs=500, m=20000, lr=1e-3):
    net=Net(6 if use_v else 5); opt=torch.optim.Adam(net.parameters(),lr=lr)
    for ep in range(epochs):
        S,v=paths_t(sampler(),m)
        loss=cvar_torch(-barrier_pnl(net,S.detach(),v.detach(),prem_P0,use_v))
        opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1)%100==0: print(f"  ep {ep+1}  CVaR train = {loss.item():.3f}")
    return net

def eval_net(net, use_v, par, prem, m=60000, seed=7):
    Snp,vnp=sim_np(par,m,seed,mu)
    with torch.no_grad():
        pnl=barrier_pnl(net,torch.tensor(Snp,dtype=torch.float32),
                        torch.tensor(vnp,dtype=torch.float32),prem,use_v).numpy()
    return cvar(pnl)

print("1) reseau qui VOIT v, entraine sur P0"); net_v = train(True, lambda: P0)
print("2) reseau AVEUGLE a v, entraine sur P0"); net_blind = train(False, lambda: P0)


In [ ]:
rng_dr=np.random.default_rng(0)
def sampler_dr():
    theta=rng_dr.uniform(0.02,0.09); xi=rng_dr.uniform(0.2,0.6); rho=rng_dr.uniform(-0.9,-0.3)
    return (theta,2.0,theta,xi,rho)
print("3) reseau AVEUGLE randomise (DR)"); net_blind_dr = train(False, sampler_dr, epochs=700)

print(f"{'scenario':12s} {'classique':>10s} {'voit v':>8s} {'aveugle':>8s} {'aveugle+DR':>11s}")
V,B,BDR={},{},{}
for name,par in scen.items():
    prem=ref[name]['prem']
    V[name]=eval_net(net_v,True,par,prem); B[name]=eval_net(net_blind,False,par,prem); BDR[name]=eval_net(net_blind_dr,False,par,prem)
    print(f"{name:12s} {ref[name]['delta']:10.2f} {V[name]:8.2f} {B[name]:8.2f} {BDR[name]:11.2f}")


In [ ]:
labels=list(scen.keys()); x=np.arange(len(labels)); wd=0.2
fig,ax=plt.subplots(figsize=(12,5))
ax.bar(x-1.5*wd,[V[k] for k in labels],wd,label='voit v (P0)',color='tab:green')
ax.bar(x-0.5*wd,[B[k] for k in labels],wd,label='aveugle a v (P0)',color='tab:red')
ax.bar(x+0.5*wd,[BDR[k] for k in labels],wd,label='aveugle + randomisation',color='tab:blue')
ax.bar(x+1.5*wd,[ref[k]['delta'] for k in labels],wd,label='classique delta BS-barriere',color='tab:orange')
ax.axhline(20.7,color='gray',ls='--',lw=1,label='ne rien faire (~20.7)')
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('CVaR 95%')
ax.set_title("Barriere : l'aveuglement a la vol casse la couverture sur S2 (knock 33%)")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()


## Ce qu'il faut retenir

- **La robustesse est une propriété du produit.** Le même aveuglement à $v$ ne coûtait presque rien au call vanille (14b : +30% sur S2) mais devrait casser le hedger de barrière, parce que la proba de knock, qui pilote toute sa politique, double avec le niveau de vol. Un réseau qui ne voit pas la vol est perdu sur un produit dont la vol change tout.
- **Le réseau qui voit $v$ reste robuste.** Confirmation du principe : tant que le réseau observe la variable d'état qui pilote la décision (ici $v$ et la distance à la barrière), il s'adapte au régime.
- **La randomisation est l'assurance quand l'information manque.** Pour le réseau aveugle, entraîner sur une plage de régimes restaure la robustesse, au prix d'un peu d'optimalité nominale.
- **La leçon d'entretien, en une phrase :** le deep hedging ne sur-ajuste pas le simulateur par nature ; il sur-ajuste seulement les dimensions de régime qu'il ne peut pas observer, et d'autant plus que le produit y est sensible. Sur un vanille, ce risque est négligeable ; sur une barrière, il est central, et on le couvre soit par un bon proxy de vol, soit par la randomisation.
